# Act 1 — Why not just use EC2 for everything?

You just spent the last notebook learning the canonical web-tier pattern on AWS — EC2 instances behind a load balancer, fronted by an Auto Scaling Group. It works. So why would you ever choose anything else?

Three pressures push real workloads off pure EC2.

The first is **bursty or unpredictable demand**. When traffic spikes randomly and falls to zero between spikes, an Auto Scaling Group ramps too slowly, and you pay for the idle floor between events. You want something that goes from zero to a thousand concurrent requests in seconds and bills nothing when it sits quiet.

The second is **operational surface**. You do not want to patch operating systems, manage AMIs, or write scaling logic for code that runs for one second once a minute. You want to hand AWS the code and let it deal with everything around it.

The third is **runtime packaging**. You want the exact same binary, with the exact same dependencies, to run on your laptop, in CI, in dev, and in production. EC2 with a custom AMI gets you there, but the AMI pipeline is heavy and AMIs are large. You want lighter — a single image you push to a registry and run anywhere.

Each pressure points somewhere. **Lambda** answers the first two — zero servers, instant scale, pay per millisecond. **Containers** answer the third — your code with its runtime in one portable unit. **Fargate** folds containers and serverless together — container portability with no host to manage.

## Serverless and containers in one frame

**Serverless** is a model where you write code and AWS handles everything else — provisioning, patching, scaling, high availability. You pay only for compute time your code actually uses, and idle costs nothing.

**Containers** are the other side of the same coin — your code, runtime, libraries, and config packaged into a single portable unit that runs anywhere a container engine does. AWS runs containers managed (ECS), Kubernetes-flavoured (EKS), or fully serverless (Fargate).

This notebook covers Lambda first — the canonical serverless compute — then the container services and the decision rules for choosing between them.

## The Serverless Trade-off

| | EC2 / Containers on EC2 | Lambda / Fargate |
|---|---|---|
| **Provisioning** | You manage capacity | AWS manages everything |
| **Scaling** | Auto Scaling group, manual triggers | Automatic, near-instant |
| **Cost when idle** | Yes — instances run 24/7 | None |
| **Cost model** | Per hour of instance time | Per request + duration |
| **Max runtime per unit of work** | Unbounded | 15 min for Lambda; unbounded for Fargate |
| **OS / runtime patching** | You | AWS |

Serverless wins on event-driven, intermittent, or unpredictable workloads. Long-running, stateful, or sustained-CPU work usually belongs on EC2 or containers — at high steady-state utilisation, per-invocation pricing becomes more expensive than per-hour.

# Act 2 — Lambda: zero servers

You hand AWS a function. AWS gives it a runtime, a network, an IAM identity, and a queue of events to react to. From the outside it looks like your code is just there, waiting — there is no server, no Auto Scaling Group, no AMI to bake. Internally it is a fleet of micro-VMs that AWS spins up on demand, but you never see them.

This act walks the Lambda model in the order the questions actually arrive. How a function runs. How you ship code into it. How you size and time-bound it. Why cold starts exist. How events reach the function — synchronously, asynchronously, or polled. How concurrency, permissions, and VPC reachability work. What happens when the handler fails. Two edge variants. And how Lambda is actually fronted as HTTP — almost always via API Gateway or a Function URL.

## AWS Lambda

Lambda runs your code in response to events, scales automatically from zero to thousands of concurrent executions, and bills per millisecond of execution time. You provide a function; AWS provides a runtime, the network, the scaling, and the integration with dozens of event sources.

A function consists of:
- **Handler** — the entry point AWS calls per invocation
- **Runtime** — Python, Node.js, Java, Go, Ruby, .NET, or a custom runtime
- **Execution role** — IAM permissions the function has at runtime
- **Configuration** — memory, timeout, environment variables, network

## Execution Environment

When an event triggers your function, Lambda finds or creates an **execution environment** — a sandboxed micro-VM (Firecracker) running your runtime and your code.

```
event → init code (once per env) → handler(event, context) → response
```

- **Init code** runs once when the environment starts — use it to set up DB connections, SDK clients, cached config, loaded ML models. It is reused across subsequent invocations on the same environment.
- The **handler** runs once per invocation. It receives the event payload and a `context` object exposing the request ID, remaining time, function name, and log group.
- After the handler returns, the environment stays warm for a period and may serve more invocations. Idle environments are eventually torn down.

The execution environment is the unit of caching, of cold-start cost, and of concurrency — every concept below ties back to it.

## Packaging and Layers

| Option | Limit | Best for |
|---|---|---|
| **Inline (zip in console)** | 3 MB source | Tiny utility functions, quick edits |
| **Zip via S3** | 250 MB unzipped | Standard deployments |
| **Container image (OCI)** | 10 GB | Large dependencies, ML models, custom runtimes |

**Lambda Layers** are versioned zip archives of shared libraries or custom runtimes that mount into the execution environment. Up to five layers per function, shared across functions and accounts. Use layers to keep your function packages small, share company-wide helpers, and avoid re-uploading large dependencies on every deploy.

## Configuration

| Setting | Range | Notes |
|---|---|---|
| **Memory** | 128 MB – 10,240 MB | CPU scales proportionally; one full vCPU at ~1,769 MB |
| **Timeout** | 1 sec – 900 sec (15 min) | Hard cap — function is killed if exceeded |
| **Ephemeral `/tmp`** | 512 MB – 10,240 MB | Survives across warm invocations in the same env |
| **Environment variables** | 4 KB total | Encrypted at rest with KMS |
| **Default account concurrency** | 1,000 per region | Soft limit — request more via support |

The memory–CPU relationship is the most important sizing knob: there is no direct CPU dial, so for CPU-bound work, raise memory until you have enough vCPU, even if you don't need the extra RAM.

## Cold Starts and Warm Starts

```
Cold:  download code → start runtime → init code → handler
Warm:                                                handler
```

A **cold start** happens on the first invocation, after an idle period, or when Lambda has to scale out to handle more concurrency. Cold start cost depends mostly on the runtime:

| Runtime | Typical cold start |
|---|---|
| Python / Node.js | 100–200 ms |
| Java / .NET | 500 ms – 1 s+ (JVM/CLR startup) |
| Container image | Variable — depends on image size, can be seconds |

**Reducing cold-start pain:**
- Keep the deployment package small
- Move heavy setup (SDK clients, DB pools) into init code so it runs once per environment, not per invocation
- Choose a lighter runtime where possible
- Use **Provisioned Concurrency** to keep N environments permanently warm

Provisioned Concurrency pre-initialises environments. Requests served by them have no cold start. You pay for provisioned capacity even when idle, so reserve it only for latency-sensitive surfaces.

## Invocation Models

Lambda is triggered three ways, and the model determines retry and error semantics.

**Synchronous** — caller waits for the response. Used by API Gateway, ALB, Function URLs, Cognito, Lex, direct SDK invokes. **No automatic retries** — the error is returned to the caller, which decides whether to retry.

**Asynchronous** — Lambda queues the event and returns immediately to the caller. Used by S3 events, SNS, EventBridge, CloudWatch Logs, SES. Lambda **retries up to 2 times** with exponential backoff on failure. Persistent failures can be routed to a Dead Letter Queue or a Lambda Destination.

**Event source mapping (poll-based)** — Lambda polls the source and invokes your function with batches of records. Used by SQS, Kinesis Data Streams, DynamoDB Streams, and self-managed Kafka or MSK.

| Source | Ordering | Scaling unit |
|---|---|---|
| SQS Standard | None | One concurrent invocation per batch; scales out to the function's concurrency cap |
| SQS FIFO | Preserved per message group | One concurrent invocation per group |
| Kinesis / DynamoDB Streams | Preserved per shard | One concurrent invocation per shard |

## Concurrency

Concurrency is the count of execution environments running your function at the same time. The account has a regional limit (default 1,000) that is shared by every function with no reservation.

**Reserved concurrency** — pinned to a specific function. It both **guarantees** the function always has at least that capacity (no other function can starve it) and **caps** the function at that limit. Set to 0 to effectively disable the function. The common architectural use is protecting a downstream resource — if a database can handle 50 connections, cap the Lambda at 50.

**Provisioned concurrency** — orthogonal to reserved. It pre-warms a specific number of environments. No cold starts on requests they handle. You pay for the provisioned count regardless of traffic, so this is the lever for latency-sensitive APIs, not for throughput.

When a function exceeds its concurrency limit, Lambda **throttles**. Synchronous callers receive an HTTP 429 immediately. Asynchronous events are queued and retried. Event source mappings back off.

## Permissions — Two Sides

Lambda uses two distinct IAM constructs:

| | Execution Role | Resource-based Policy |
|---|---|---|
| **Controls** | What the function can do | Who can invoke the function |
| **Attached to** | The function (assumed at runtime) | The function as a resource |
| **Example** | `dynamodb:PutItem` on a specific table | Allow API Gateway to invoke this function |

The execution role is the IAM role Lambda assumes when running your code — the credentials inside the handler are role credentials. Apply least privilege rigorously.

Resource-based policies are added automatically when you wire a trigger in the console (API Gateway, S3, EventBridge), and you set them manually for cross-account invocation patterns.

## VPC Configuration

By default Lambda runs **outside your VPC** in an AWS-managed network with internet access. To reach **private** resources — RDS in a private subnet, ElastiCache, an internal service — attach the function to a VPC subnet.

```
Lambda outside VPC: internet ✓   private VPC resources ✗
Lambda in VPC:      internet ✗ (unless NAT)   private VPC resources ✓
```

Internet access from a VPC-attached Lambda requires routing through a NAT Gateway, exactly as for an EC2 instance in a private subnet.

Older Lambda versions paid a heavy cold-start penalty for ENI attachment. AWS now uses **Hyperplane ENIs** — shared across functions in the same subnet — and the VPC cold-start overhead is negligible. The trade-off today is mostly about reachability, not latency.

## Error Handling and Destinations

**Async invocations** support two failure pathways:

- **Dead Letter Queue** — an SQS queue or SNS topic that receives the failed event payload after retries exhausted. Older, simpler, payload only.
- **Lambda Destinations** — route both success and failure to SQS, SNS, EventBridge, or another Lambda. Carries the full event plus the function response, which is richer for debugging.

**Event source mappings** (SQS, Kinesis, DynamoDB Streams) have their own error semantics:
| Source | Retry behaviour |
|---|---|
| SQS | Retries until the message visibility timeout expires or the receive count hits the DLQ trigger |
| Kinesis / DynamoDB Streams | Retries until the record expires (default 24 h) or bisect-on-error narrows the failing record |

Partial batch responses (`ReportBatchItemFailures`) let your function report which records in a batch failed, so successful records aren't reprocessed.

## Lambda@Edge and CloudFront Functions

Both run code at CloudFront edge locations, closer to users than your origin region. They differ in capability and cost.

| | Lambda@Edge | CloudFront Functions |
|---|---|---|
| **Runtime** | Node.js, Python | JavaScript only |
| **Max duration** | 5 s (viewer events) / 30 s (origin events) | < 1 ms |
| **Network access** | Yes — call other services | None |
| **Use for** | Auth, A/B routing, header rewrites with API calls | Pure header / URL transforms, simple redirects |

CloudFront Functions are the cheaper, faster default for header manipulation and URL rewrites at the edge. Reach for Lambda@Edge when the logic needs to make a network call or run a richer runtime.

## API Gateway and Function URLs

Lambda needs something in front of it to handle HTTP. Two options:

**Lambda Function URLs** are a built-in HTTPS endpoint directly on a function. One URL per function, no extra service, IAM or open auth. Use them for simple internal services, webhooks, or quick prototypes.

**API Gateway** is the full HTTP front door for Lambda — and the canonical pairing for production serverless APIs. It adds routing, throttling, API keys and usage plans, request/response transformation, validation, caching, custom domains, WAF integration, and authentication via Cognito or Lambda authorisers.

| | Function URL | API Gateway HTTP API | API Gateway REST API |
|---|---|---|---|
| **Setup** | One toggle | Simple | Most configurable |
| **Auth** | IAM or open | JWT (Cognito), IAM | Cognito, IAM, Lambda authoriser |
| **Throttling / quotas** | No | Per-route | Per-stage and per-key |
| **Caching** | No | No | Yes |
| **Cost** | Lowest | Lower | Highest |

The decision rule: Function URLs for the simplest case, HTTP APIs for most production REST workloads, REST APIs only when you need request/response transformation, caching, or fine-grained throttling.

## The Serverless Stack

Lambda is rarely alone. A typical serverless application combines:

| Service | Role |
|---|---|
| **API Gateway** | HTTP/WebSocket front door |
| **DynamoDB** | Serverless NoSQL data store, scales to zero |
| **S3** | Object storage, also a Lambda event source |
| **SQS / SNS / EventBridge** | Decoupling, fan-out, event routing |
| **Step Functions** | Multi-step workflow orchestration |
| **Aurora Serverless v2** | Serverless relational DB |
| **Cognito** | User identity, sign-in, JWT issuance |

The whole stack scales independently and bills per use — which is the underlying value proposition of going serverless in the first place.

In [ ]:
import boto3, json, zipfile, io

lam = boto3.client("lambda", region_name="us-east-1")

source = b'''
import json
def handler(event, context):
    name = event.get("name", "World")
    return {"statusCode": 200, "body": json.dumps({"message": f"Hello, {name}!"})}
'''

buf = io.BytesIO()
with zipfile.ZipFile(buf, "w") as zf:
    zf.writestr("handler.py", source)
buf.seek(0)

lam.create_function(
    FunctionName="demo-hello",
    Runtime="python3.12",
    Role="arn:aws:iam::111111111111:role/lambda-execution-role",
    Handler="handler.handler",
    Code={"ZipFile": buf.read()},
    Timeout=30,
    MemorySize=256,
    Environment={"Variables": {"STAGE": "dev"}},
)

# Reserved concurrency cap — protects a downstream DB
lam.put_function_concurrency(FunctionName="demo-hello", ReservedConcurrentExecutions=50)

# Synchronous invoke
resp = lam.invoke(FunctionName="demo-hello", Payload=json.dumps({"name": "Alice"}))
print(json.loads(resp["Payload"].read()))

# Act 3 — Containers: portable runtime

Lambda gives you zero servers but constrains your runtime to AWS's choices — its Python version, its library set, its execution model. What if you need a specific Java release, a custom native binary, or a runtime AWS does not ship at all? What if you simply want the exact same image to run unchanged on your laptop, in CI, and in production?

That is what containers solve. A container is your code, its runtime, and its dependencies packaged into one image that runs identically wherever a container engine exists.

AWS runs containers through two orchestrators. **ECS** is AWS's native orchestrator — the simplest path if you have no prior Kubernetes investment. **EKS** is managed Kubernetes for teams that already speak Kubernetes or need its ecosystem. Under either orchestrator, the compute that actually runs your containers can be EC2 instances you manage, or **Fargate** — AWS-managed compute where the host is invisible. This act covers ECS in depth; the next visits EKS.

## Containers — the Other Compute Model

A container packages your application, its runtime, and its dependencies into one image that runs identically wherever a container engine is present. Containers boot in seconds, share the host kernel rather than virtualising hardware, and form the unit of deployment for almost every modern microservice architecture.

The container model fits where Lambda doesn't:
- Long-running processes beyond Lambda's 15-minute cap
- Sustained CPU or memory utilisation where per-invocation pricing loses to per-hour
- Workloads needing finer control over the runtime than Lambda exposes
- Existing applications that are already containerised

AWS runs containers three ways: **ECS** for AWS-native orchestration, **EKS** for managed Kubernetes, and **Fargate** as the serverless compute engine underneath either of them.

## Amazon ECR

**ECR** (Elastic Container Registry) is AWS's managed container image registry — the private equivalent of Docker Hub, integrated with IAM.

| Capability | Detail |
|---|---|
| **Private registry** | One per account per region, IAM-scoped |
| **Public registry** | `public.ecr.aws` for sharing images publicly |
| **Image scanning** | Basic (on push) or Enhanced (continuous, via Inspector) |
| **Lifecycle policies** | Expire old image versions automatically |
| **Replication** | Cross-region and cross-account |
| **Encryption** | At-rest encryption via KMS |

Authentication uses a short-lived token from `aws ecr get-login-password` — there are no long-lived registry credentials to manage.

## Amazon ECS — Object Model

**ECS** is AWS's native container orchestrator. It schedules, places, scales, and replaces containers across a cluster.

```
Cluster
  ├── Service A (desired: 3)        ├── Service B (desired: 2)
  │     ├── Task                    │     ├── Task
  │     ├── Task                    │     └── Task
  │     └── Task
```

| Object | What it is |
|---|---|
| **Cluster** | Logical fleet, backed by EC2 capacity or by Fargate |
| **Task Definition** | Blueprint — containers, CPU/memory, image, ports, volumes, IAM roles |
| **Task** | A running instance of a task definition (one or more containers together) |
| **Service** | Maintains a desired number of tasks, integrates with a load balancer, handles replacement on failure |

The unit of deployment is the **task**; the unit of long-running operation is the **service**. One-off batch jobs run as standalone tasks.

## Task Definitions and the Two Roles

A task definition is a JSON document that pins down everything about how a task runs: image, CPU and memory, port mappings, environment variables, log driver, volumes, networking mode, and two distinct IAM roles.

| Role | Used by | Typical permissions |
|---|---|---|
| **Task Execution Role** | The ECS agent on the host | `ecr:GetAuthorizationToken`, `ecr:BatchGetImage`, `logs:CreateLogStream`, `logs:PutLogEvents` |
| **Task Role** | Your container's application code | Whatever your app needs — `s3:GetObject`, `dynamodb:Query`, etc. |

Splitting them is deliberate. The execution role is plumbing — the agent uses it to pull images and ship logs. The task role is your application's identity — credentials are exposed to the container via the standard AWS SDK metadata path, so `boto3` inside the container just works with no embedded keys.

## EC2 vs Fargate Launch Types

ECS supports two launch types for the same task definition — the difference is who owns the host.

| | EC2 launch type | Fargate launch type |
|---|---|---|
| **Hosts** | EC2 instances you (or an ASG) manage | AWS — invisible |
| **Visibility** | Choose instance type, AMI, SSH in | None |
| **Cost model** | Per-instance hour | Per task vCPU + memory per second |
| **Idle cost** | Yes — paid even with zero tasks | None |
| **Maintenance** | Patch, scale, secure the fleet | Nothing |
| **Choose when** | High steady-state utilisation, GPUs, special instance types, custom kernel | Variable workloads, microservices, no infra to manage |

**Fargate sizing** uses discrete vCPU/memory combinations: 0.25 vCPU with 0.5–2 GB, up to 16 vCPU with 32–120 GB. You pick a valid pair; AWS picks the host.

The general rule: start with **Fargate** for simplicity. Move to **EC2 launch type with Savings Plans** when you've grown into steady, predictable utilisation and the per-hour math beats per-second.

## ECS Networking

Each task can use one of four network modes — but for almost all new workloads, the answer is `awsvpc`.

| Mode | How it works | Use |
|---|---|---|
| **awsvpc** | Each task gets its own ENI and private IP in your VPC | Required for Fargate; default for EC2 |
| **bridge** | Docker's NAT bridge on the host, dynamic ports | EC2 only, legacy |
| **host** | Container shares the host's network namespace | EC2 only, high-throughput |
| **none** | No external network | Isolated tasks |

With `awsvpc`, each task carries its own security group and ENI — treat tasks as you would small EC2 instances on the network. Dynamic port mapping is no longer needed because every task has its own IP.

**Load balancer integration:** attach the ECS service to an ALB or NLB target group. ECS registers tasks on launch and deregisters on stop, respecting deregistration delay so in-flight requests complete.

## ECS Service Auto Scaling

ECS services scale via **Application Auto Scaling**, the same engine that scales DynamoDB capacity and Aurora Serverless. The policy types mirror ASG scaling:

- **Target tracking** — keep average CPU at 70%, or requests-per-target at 1,000
- **Step scaling** — different magnitudes of response at different alarm thresholds
- **Scheduled scaling** — predictable patterns

For the **EC2 launch type** you also need to scale the underlying EC2 capacity — use an **ECS Capacity Provider** backed by an ASG to automate this so the cluster grows and shrinks with the services on it. Fargate has no underlying capacity to manage, which is much of its appeal.

# Act 4 — When ECS is not enough

ECS works for the vast majority of new container workloads on AWS. But three situations push you toward EKS instead.

The first is an **existing Kubernetes investment** — your team already runs Kubernetes elsewhere and wants one operational model across environments. The second is **multi-cloud or on-premises portability** — Kubernetes manifests are largely cloud-agnostic; ECS task definitions are not. The third is **rich Kubernetes-specific scheduling and ecosystem needs** — taints, tolerations, custom controllers, Helm charts, service meshes, GitOps tooling.

EKS is more complex and more expensive at small scale than ECS. The complexity earns its place when Kubernetes is a real requirement, not a preference.

## Amazon EKS — Managed Kubernetes

**EKS** runs the Kubernetes control plane — API server, etcd, scheduler, controller manager — as a managed AWS service. You're responsible for worker nodes (or use Fargate for the nodeless pattern).

Choose EKS over ECS when one of these applies:
- The team already runs Kubernetes elsewhere and you want one operational model
- Multi-cloud portability matters — Kubernetes manifests are largely cloud-agnostic
- You need rich Kubernetes-specific scheduling (taints, tolerations, custom schedulers, operators)
- You're adopting the broader Kubernetes ecosystem — Helm charts, service meshes, GitOps tools

Choose **ECS** when none of those apply and you want simplicity, lower cost at small scale, and the deepest AWS-native integration. The EKS control plane costs ~$0.10/hour per cluster, which matters at small scale but disappears at large scale.

## EKS Node Options

| Node type | Who manages the EC2 lifecycle | Best for |
|---|---|---|
| **Managed Node Group** | AWS — launch, drain, update, terminate | Most clusters; sensible default |
| **Self-Managed Nodes** | You — your AMI, your ASG | Custom AMIs, Windows, kernel-level requirements |
| **Fargate Profile** | AWS — no EC2 at all | Burstable microservices, no node ops |

**EKS Fargate** runs pods matching a profile selector (namespace + labels) on Fargate, with no nodes to patch or scale. The trade-offs are real:
- No DaemonSets — no per-node agents
- No privileged containers
- No EBS-backed persistent volumes (EFS works)
- No GPU support

For most clusters, Managed Node Groups are the right default. Add Fargate profiles for specific workloads where node-management overhead isn't worth it.

## ECS vs EKS — Choosing

| Question | Lean ECS | Lean EKS |
|---|---|---|
| New to containers on AWS, no K8s history? | ✓ | |
| Existing Kubernetes workloads or manifests? | | ✓ |
| Need multi-cloud or on-prem portability? | | ✓ |
| Want deepest AWS-native integration with minimal complexity? | ✓ | |
| Need rich Kubernetes scheduler features (taints, custom controllers)? | | ✓ |
| Cluster cost matters at small scale? | ✓ | |
| Comfortable owning Kubernetes operational complexity? | | ✓ |

The default for new AWS-only workloads with no existing Kubernetes investment is **ECS on Fargate** — it's the simplest, cheapest, lowest-operational-overhead path. EKS earns its complexity premium when Kubernetes is a real requirement, not a preference.

# Act 5 — The full picture

Lambda, ECS, and EKS are the three large compute models on AWS. But there is one higher-level option that hides most of the choice for you, and you still need a way to pick among everything at the end.

**App Runner** is the highest-level container deployment path. Point it at an image or a Git repo and it builds, deploys, load-balances, terminates TLS, and scales. It is the right choice for a small-to-medium web service that just needs to be online. You outgrow it when the architecture grows past one service or needs custom networking.

The closer for this notebook is a decision tree — given a workload shape, which compute service to reach for — followed by a worked example of standing up a Fargate service end to end.

## AWS App Runner

**App Runner** is the highest-level container deployment service on AWS. Point it at an ECR image or a Git repository; it builds, deploys, load-balances, terminates TLS, scales, and runs health checks for you.

| | App Runner | ECS on Fargate | Lambda |
|---|---|---|---|
| **Setup** | Minimal — image URL or repo URL | Cluster, task def, service, ALB | Function + trigger |
| **Max request duration** | None | None | 15 min |
| **Cold starts** | Minimal | None (tasks are persistent) | Yes |
| **VPC access** | Optional | Yes | Optional |
| **Best for** | Simple web APIs, single-service apps | Production microservices at scale | Event-driven, short tasks |

App Runner's sweet spot is the small-to-medium web service that just needs to be online behind HTTPS with autoscaling. As architectural complexity grows — multiple services, custom networking, advanced load balancing — you outgrow App Runner and move down to ECS or EKS.

## Choosing a Compute Service

A practical decision tree for new workloads:

| Workload shape | Reach for |
|---|---|
| Event-driven, short bursts, < 15 min per unit | **Lambda** |
| HTTP API with simple routing, predictable shape | **Lambda + API Gateway** or **App Runner** |
| Long-running service, sustained traffic, no Kubernetes need | **ECS on Fargate** |
| Long-running service at high steady utilisation, cost-sensitive | **ECS on EC2** with Savings Plans |
| Existing Kubernetes workloads, multi-cloud, advanced scheduling | **EKS** |
| Full OS control, special hardware, custom kernel | **EC2 directly** |

The progression from Lambda to App Runner to ECS-Fargate to EKS is roughly a ladder of increasing control and increasing operational responsibility. Pick the highest rung that meets your requirements — every step down adds work you have to justify.

In [ ]:
import boto3

ecs = boto3.client("ecs", region_name="us-east-1")
ecr = boto3.client("ecr", region_name="us-east-1")

# Repository for the image
repo = ecr.create_repository(
    repositoryName="my-app",
    imageScanningConfiguration={"scanOnPush": True},
    encryptionConfiguration={"encryptionType": "KMS"},
)["repository"]

# Task definition — Fargate, awsvpc, two distinct IAM roles
task_def = ecs.register_task_definition(
    family="web-app",
    networkMode="awsvpc",
    requiresCompatibilities=["FARGATE"],
    cpu="512", memory="1024",
    executionRoleArn="arn:aws:iam::111111111111:role/ecsTaskExecutionRole",
    taskRoleArn="arn:aws:iam::111111111111:role/webAppTaskRole",
    containerDefinitions=[{
        "name": "web",
        "image": f"{repo['repositoryUri']}:latest",
        "portMappings": [{"containerPort": 80, "protocol": "tcp"}],
        "essential": True,
        "logConfiguration": {
            "logDriver": "awslogs",
            "options": {
                "awslogs-group": "/ecs/web-app",
                "awslogs-region": "us-east-1",
                "awslogs-stream-prefix": "web",
            },
        },
    }],
)["taskDefinition"]

# Cluster + service behind an ALB target group
ecs.create_cluster(clusterName="demo-cluster")
ecs.create_service(
    cluster="demo-cluster",
    serviceName="web-service",
    taskDefinition="web-app",
    desiredCount=2,
    launchType="FARGATE",
    networkConfiguration={"awsvpcConfiguration": {
        "subnets": ["subnet-aaa", "subnet-bbb"],
        "securityGroups": ["sg-xxxxxxxx"],
        "assignPublicIp": "DISABLED",
    }},
    loadBalancers=[{
        "targetGroupArn": "arn:aws:elasticloadbalancing:...",
        "containerName": "web",
        "containerPort": 80,
    }],
)